# Fed rate decisions — next 6 months (Polymarket implied probabilities)

Pulls FOMC-decision markets from Polymarket. Where several markets describe the same meeting, we keep only the deepest (max liquidity) to cut noise.

In [ ]:
import pandas as pd

from alpha_lab.data.loaders.polymarket import (
    implied_prob,
    search_events,
    search_markets,
    tidy,
    top_by_liquidity,
)

HORIZON_MONTHS = 6
NOW = pd.Timestamp.utcnow()
HORIZON_END = NOW + pd.DateOffset(months=HORIZON_MONTHS)

In [ ]:
# Cast a wide net: FOMC / Fed / rate-decision phrasing varies market-to-market.
queries = ["fed decision", "fed rate", "fomc", "interest rate"]
frames = [search_markets(q, limit=200) for q in queries]
markets = (
    pd.concat(frames, ignore_index=True)
    .drop_duplicates(subset=["id"])
)
markets = markets[markets["endDate"].between(NOW, HORIZON_END)]
len(markets)

In [ ]:
# Bucket by FOMC meeting end-date (yyyy-mm), keep the deepest market per bucket.
markets["meeting"] = markets["endDate"].dt.strftime("%Y-%m")
per_meeting = (
    markets.groupby("meeting", group_keys=False)
    .apply(lambda g: top_by_liquidity(g, n=1))
    .sort_values("endDate")
)
tidy(per_meeting)

In [ ]:
# For each selected meeting, pull sibling markets in the same event to see the
# full probability distribution over rate outcomes (cut / hold / hike / by-how-much).
event_slugs = per_meeting["eventSlug"].dropna().unique().tolist()

dists = []
for slug in event_slugs:
    ev = search_events(query=slug, limit=5)
    if ev.empty:
        continue
    row = ev.iloc[0]
    children = pd.DataFrame(row.get("markets") or [])
    if children.empty:
        continue
    children = children.assign(
        event=row.get("title"),
        yes=children.apply(lambda r: implied_prob(r.to_dict(), "Yes"), axis=1),
    )
    dists.append(children[["event", "question", "yes", "liquidityNum", "volumeNum"]])

distribution = pd.concat(dists, ignore_index=True) if dists else pd.DataFrame()
distribution